In [1]:
import os
os.chdir('..')


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, LogitsProcessor, LogitsProcessorList
import torch
import json
from tqdm import tqdm
import argparse
import re

/raid/home/m13521157/absa-sft-comparison/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
os.environ["CUDA_VISIBLE_DEVICES"] = "5,7"

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
model_path = "outputs/models/checkpoint-14470-indolegoabsa-multi"
tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side="left")
ori_model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B', torch_dtype=torch.bfloat16, device_map="auto", cache_dir=os.getenv("HF_CACHE_DIR"))
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map="auto")

`torch_dtype` is deprecated! Use `dtype` instead!


In [6]:
test_data_path = 'dataset/hoasa_hotel/indo/legoabsa_multitask/test.json'
with open(test_data_path, 'r') as f:
	test_data = json.load(f)
test_data[0]

{'sentence_id': 3500,
 'instance_id': 15145,
 'task_elements': 'aos',
 'input': 'pelayanan nya sangat ramah .| aspect: <|aspect|> , opinion: <|opinion|> , sentiment: <|sentiment|>',
 'target': '<|aspect|> pelayanan nya <|opinion|> sangat ramah <|sentiment|> positive',
 'element_order': 'aos',
 'dataset_type': 'hotel_reviews'}

In [7]:
check_inputs = [f'{test_data[i]['input']} =>' for i in range(5)]
check_labels = [f'{test_data[i]["target"]}' for i in range(5)]
check_labels

['<|aspect|> pelayanan nya <|opinion|> sangat ramah <|sentiment|> positive',
 '<|aspect|> wifi <|opinion|> tidak bagus harus keluar kamar <|sentiment|> negative',
 '<|aspect|> kamarnya <|opinion|> beda <|sentiment|> negative',
 '<|aspect|> over all <|opinion|> baik <|sentiment|> positive;<|aspect|> air hot waternya <|opinion|> akan lebih memuaskan jika air hot waternya bisa nyala 24jam <|sentiment|> negative',
 '<|aspect|> fasilatas <|opinion|> sesuia <|sentiment|> positive']

In [8]:
tokenized_inputs = tokenizer(check_inputs, return_tensors='pt', padding=True, truncation=True).to(model.device)

In [9]:
tokenizer.special_tokens_map

{'eos_token': '<|endoftext|>',
 'pad_token': '<|endoftext|>',
 'additional_special_tokens': ['<|im_start|>',
  '<|im_end|>',
  '<|object_ref_start|>',
  '<|object_ref_end|>',
  '<|box_start|>',
  '<|box_end|>',
  '<|quad_start|>',
  '<|quad_end|>',
  '<|vision_start|>',
  '<|vision_end|>',
  '<|vision_pad|>',
  '<|image_pad|>',
  '<|video_pad|>',
  '<|aspect|>',
  '<|opinion|>',
  '<|sentiment|>']}

In [10]:
# Decode the first input back to list of tokens
detokenized_inputs = tokenizer.decode(tokenized_inputs['input_ids'][0], skip_special_tokens=False)
detokenized_inputs

'<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>pelayanan nya sangat ramah .| aspect: <|aspect|> , opinion: <|opinion|> , sentiment: <|sentiment|> =>'

In [11]:
# Detokenize to list of tokens
tokens = tokenizer.convert_ids_to_tokens(tokenized_inputs['input_ids'][0])
tokens

['<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 '<|endoftext|>',
 'p',
 'elay',
 'anan',
 'Ġnya',
 'Ġsangat',
 'Ġram',
 'ah',
 'Ġ.',
 '|',
 'Ġaspect',
 ':',
 'Ġ',
 '<|aspect|>',
 'Ġ,',
 'Ġopinion',
 ':',
 'Ġ',
 '<|opinion|>',
 'Ġ,',
 'Ġsentiment',
 ':',
 'Ġ',
 '<|sentiment|>',
 'Ġ=>']

In [12]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151668, 896, padding_idx=151643)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
   

In [13]:
ori_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [14]:
# Add special tokens for LEGO-ABSA if needed
if 'legoabsa' in model_path:
	print("Adding LEGO-ABSA special tokens to tokenizer...")
	custom_tokens = ["<|aspect|>", "<|opinion|>", "<|sentiment|>"]

	# Check if the custom tokens is already added to the tokenizer or not
	if all(token in tokenizer.special_tokens_map['additional_special_tokens'] for token in custom_tokens):
		print("Custom tokens already exist in the tokenizer. Skipping addition.")
	else:
		print('Old tokenizer size:', len(tokenizer))

		num_added_tokens = tokenizer.add_special_tokens({
			"additional_special_tokens": tokenizer.special_tokens_map['additional_special_tokens'] + custom_tokens
		})

		print(f"Added {num_added_tokens} new tokens.")
		print(f"New tokenizer size: {len(tokenizer)}")

Adding LEGO-ABSA special tokens to tokenizer...
Custom tokens already exist in the tokenizer. Skipping addition.


In [24]:
from glob import glob
model_paths_legoabsa = glob('outputs/models/hoasa_hotel/indo/*egoabsa*/seed_*/*/checkpoint-*')
len(model_paths_legoabsa)
model_paths_legoabsa

['outputs/models/hoasa_hotel/indo/legoabsa_tasktransfer/seed_31415/20251128_043831_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-4632',
 'outputs/models/hoasa_hotel/indo/legoabsa_tasktransfer/seed_31415/20251128_043831_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-4053',
 'outputs/models/hoasa_hotel/indo/legoabsa_tasktransfer/seed_31415/20251128_043831_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-5211',
 'outputs/models/hoasa_hotel/indo/legoabsa_tasktransfer/seed_31415/20251128_043831_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-2316',
 'outputs/models/hoasa_hotel/indo/legoabsa_tasktransfer/seed_31415/20251128_043831_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-3474',
 'outputs/models/hoasa_hotel/indo/legoabsa_tasktransfer/seed_31415/20251128_043831_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-2895',
 'outputs/models/hoasa_hotel/indo/legoabsa_tasktransfer/seed_31415/20251128_043831_train_model-Qwen2

In [25]:
len(tokenizer)

151668

In [15]:
tokenizer.save_pretrained('new_tokenizer')

('new_tokenizer/tokenizer_config.json',
 'new_tokenizer/special_tokens_map.json',
 'new_tokenizer/chat_template.jinja',
 'new_tokenizer/vocab.json',
 'new_tokenizer/merges.txt',
 'new_tokenizer/added_tokens.json',
 'new_tokenizer/tokenizer.json')

In [26]:
# Save the new tokenizer
for model_path in model_paths_legoabsa:
	print(f"Saving tokenizer for model at {model_path}...")
	tokenizer.save_pretrained(model_path)
# tokenizer.save_pretrained('outputs/models/hoasa_hotel/indo/legoabsa_multitask/seed_123/20251127_213816_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-8680')

Saving tokenizer for model at outputs/models/hoasa_hotel/indo/legoabsa_tasktransfer/seed_31415/20251128_043831_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-4632...
Saving tokenizer for model at outputs/models/hoasa_hotel/indo/legoabsa_tasktransfer/seed_31415/20251128_043831_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-4053...
Saving tokenizer for model at outputs/models/hoasa_hotel/indo/legoabsa_tasktransfer/seed_31415/20251128_043831_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-5211...
Saving tokenizer for model at outputs/models/hoasa_hotel/indo/legoabsa_tasktransfer/seed_31415/20251128_043831_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-2316...
Saving tokenizer for model at outputs/models/hoasa_hotel/indo/legoabsa_tasktransfer/seed_31415/20251128_043831_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-3474...
Saving tokenizer for model at outputs/models/hoasa_hotel/indo/legoabsa_tasktransfer/seed_31415/20251128_0

In [ ]:
from transformers import LogitsProcessor, AutoTokenizer
import torch

# Re-implemented and modified from [MvP: Multi-view Prompting Improves Aspect Sentiment Tuple Prediction](https://aclanthology.org/2023.acl-long.240/)

class BaseConstrainedDecoder(LogitsProcessor):
	'''
	A custom logits processor that modifies the logits during generation.
	All tokens that are not in the input sequence will be masked out (set to -inf) to prevent the model from generating them.
	Exception for special tokens like eos_token, etc.
	Can handle both batched and unbatched input.
	'''
	def __init__(self, input_ids: torch.LongTensor, tokenizer: AutoTokenizer):
		super().__init__()
		self.input_ids = input_ids
		self.tokenizer = tokenizer
		self.special_token_ids = set(tokenizer.all_special_ids)
		self.special_words = set(tokenizer(['positive', 'negative', ' positive', ' negative', 'null', ' null'], add_special_tokens=False, return_tensors='pt', padding=True, truncation=True)['input_ids'].reshape(-1).tolist())
	
	def _prepare_batch_allowed_tokens(self, tokenizer: AutoTokenizer) -> None:
		
		# Pre-compute allowed token ids for each batch item
		if self.input_ids.dim() == 1:
			# Unbatched: add batch dimension
			self.input_ids = self.input_ids.unsqueeze(0)
		
		# Create a set of allowed tokens for each batch item
		self.batch_allowed_tokens = []
		for i in range(self.input_ids.size(0)):

			# Handle spaced first word
			list_of_tokens = self.input_ids[i].tolist()
			words = tokenizer.decode(list_of_tokens, skip_special_tokens=True)
			words_split = words.split(' ')
			spaced_first_word = f' {words_split[0]}'
			spaced_token_id = set(tokenizer(spaced_first_word, add_special_tokens=False)['input_ids'])

			# Combine all allowed tokens
			allowed_tokens = set(list_of_tokens).union(self.special_token_ids).union(self.special_words).union(spaced_token_id)

			# Store the allowed tokens for this batch item
			self.batch_allowed_tokens.append(allowed_tokens)
	
	def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
		batch_size = scores.size(0)
		
		# Create a mask initialized to -inf for all tokens
		mask = torch.full_like(scores, float('-inf'))
		
		# Apply mask for each batch item
		for i in range(batch_size):
			# Handle case where batch size might be smaller than pre-computed
			allowed_idx = min(i, len(self.batch_allowed_tokens) - 1)
			allowed_token_ids = self.batch_allowed_tokens[allowed_idx]
			
			# Convert allowed token IDs to a list and set their mask values to 0
			allowed_ids_list = list(allowed_token_ids)
			mask[i, allowed_ids_list] = 0.0
		
		# Apply the mask to the scores
		modified_scores = scores + mask
		return modified_scores

class MVPConstrainedDecoder(BaseConstrainedDecoder):
	'''
	A custom logits processor for the MVP dataset that modifies the logits during generation.
	All tokens that are not in the input sequence will be masked out (set to -inf) to prevent the model from generating them.
	Exception for special tokens like eos_token, etc.
	Can handle both batched and unbatched input.
	'''
	def __init__(self, input_ids: torch.LongTensor, tokenizer: AutoTokenizer):
		super().__init__(input_ids, tokenizer)
		self.special_words = self.special_words.union(set(tokenizer([' [SSEP] '], add_special_tokens=False, return_tensors='pt', padding=True, truncation=True)['input_ids'].reshape(-1).tolist()))
		self._prepare_batch_allowed_tokens(tokenizer)

class GASConstrainedDecoder(BaseConstrainedDecoder):
	'''
	A custom logits processor for the GAS dataset that modifies the logits during generation.
	All tokens that are not in the input sequence will be masked out (set to -inf) to prevent the model from generating them.
	Exception for special tokens like eos_token, etc.
	Can handle both batched and unbatched input.
	'''
	def __init__(self, input_ids: torch.LongTensor, tokenizer: AutoTokenizer):
		super().__init__(input_ids, tokenizer)
		temp_special_words = ['|' , ' |', '| ', ' | ', ';' , ' ;', '; ', ' ; ', '(', ' (', '( ', ' ( ', ')', ' )', ') ', ' ) ']
		self.special_words = self.special_words.union(set(tokenizer(temp_special_words, add_special_tokens=False, return_tensors='pt', padding=True, truncation=True)['input_ids'].reshape(-1).tolist()))
		self._prepare_batch_allowed_tokens(tokenizer)

class LegoABSAConstrainedDecoder(BaseConstrainedDecoder):
	'''
	A custom logits processor for the LegoABSA dataset that modifies the logits during generation.
	All tokens that are not in the input sequence will be masked out (set to -inf) to prevent the model from generating them.
	Exception for special tokens like eos_token, etc.
	Can handle both batched and unbatched input.
	'''
	def __init__(self, input_ids: torch.LongTensor, tokenizer: AutoTokenizer):
		super().__init__(input_ids, tokenizer)
		self.special_words = self.special_words.union(set(tokenizer(['<|aspect|>', '<|opinion|>', '<|sentiment|>', ' ', ';'], add_special_tokens=False, return_tensors='pt', padding=True, truncation=True)['input_ids'].reshape(-1).tolist()))
		self._prepare_batch_allowed_tokens(tokenizer)

In [ ]:
custom_processor = MVPConstrainedDecoder(tokenized_inputs['input_ids'], tokenizer)
processor_list = LogitsProcessorList([custom_processor])
outputs = model.generate(
    **tokenized_inputs,
    max_new_tokens=50,
    # logits_processor=processor_list
)
outputs_logits_processor = model.generate(
	**tokenized_inputs,
	max_new_tokens=50,
	logits_processor=processor_list
)
generated_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)
generated_texts_logits_processor = tokenizer.batch_decode(outputs_logits_processor, skip_special_tokens=True)
assert len(generated_texts) == len(generated_texts_logits_processor)
for i, text in enumerate(generated_texts):
	with_lp = generated_texts_logits_processor[i].split('=>')[-1].strip()
	without_lp = text.split('=>')[-1].strip()
	print(f"Generated {i+1} without logits processor: {without_lp}")
	print(f"Generated {i+1} with logits processor: {with_lp}")
	print(f"Expected label: {check_labels[i]} | with lp: {with_lp == check_labels[i]} | without lp: {without_lp == check_labels[i]}")
	print("-" * 50)

Generated 1 without logits processor: [A] pelayanan nya [O] sangat ramah [S] positive
Generated 1 with logits processor: [A] pelayanan nya [O] sangat ramah [S] positive
Expected label: [A] pelayanan nya [O] sangat ramah [S] positive | with lp: True | without lp: True
--------------------------------------------------
Generated 2 without logits processor: [A] wifi [O] tidak bagus harus keluar kamar [S] negative
Generated 2 with logits processor: [A] wifi [O] tidak bagus harus keluar kamar [S] negative
Expected label: [A] wifi [O]  tidak bagus harus keluar kamar [S] negative | with lp: False | without lp: False
--------------------------------------------------
Generated 3 without logits processor: [A] twin bed [O] twin bed , tetapi yang ada kamarnya beda [S] negative
Generated 3 with logits processor: [A] twin bed [O] twin bed , tetapi yang ada kamarnya beda [S] negative
Expected label: [A] kamarnya [O] beda [S] negative | with lp: False | without lp: False
-----------------------------